# Week 30, Day 3: Inactivation, recovery and the refractory period

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w030/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The availability curve

For prepulse voltages from -120 to -20 mV, assume h and m have settled to their steady values. Compute the peak sodium conductance during a 10 ms test step to 0 mV, store the normalised peaks in `rel`, and fit 1 / (1 + exp((V - V_half) / k)) by grid search.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rates(V):
    V = V + 1e-7
    an = 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10)); bn = 0.125 * np.exp(-(V + 65) / 80)
    am = 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10)); bm = 4.0 * np.exp(-(V + 65) / 18)
    ah = 0.07 * np.exp(-(V + 65) / 20); bh = 1 / (1 + np.exp(-(V + 35) / 10))
    return {"n": (an, bn), "m": (am, bm), "h": (ah, bh)}

def inf_tau(V, g):
    a, b = rates(V)[g]
    return a / (a + b), 1 / (a + b)

t = np.arange(0, 10, 0.01)

def relax(x0, V, g):
    xinf, tau = inf_tau(V, g)
    return xinf - (xinf - x0) * np.exp(-t / tau)

Vpre = np.arange(-120, -19, 5.0)
peaks = np.array([(120 * relax(inf_tau(v, "m")[0], 0.0, "m") ** 3 * relax(inf_tau(v, "h")[0], 0.0, "h")).max() for v in Vpre])
rel = ...
best = (np.inf, None, None)
for vh in np.arange(-90, -30.01, 0.5):
    for k in np.arange(2, 15.01, 0.1):
        err = ...
        if err < best[0]:
            best = (err, vh, k)
_, V_half, k_slope = best
print(f"half-inactivation {V_half:.1f} mV, slope factor {k_slope:.1f} mV;  available at -65 mV: {np.interp(-65, Vpre, rel):.2f}, at -55 mV: {np.interp(-55, Vpre, rel):.2f}")
plt.plot(Vpre, rel, "o"); plt.plot(Vpre, 1 / (1 + np.exp((Vpre - V_half) / k_slope)))
plt.xlabel("prepulse voltage (mV)"); plt.ylabel("fraction available"); plt.show()

In [ ]:
import numpy as np
def test_peaks():
    "rel is the peak test conductance after each prepulse, divided by the largest"
    assert np.allclose(rel, peaks / peaks.max()) and abs(rel[0] - 1) < 1e-3

def test_fit():
    "the fitted half-inactivation voltage is between -68 and -58 mV and the slope factor between 5 and 10 mV"
    assert -68 < V_half < -58 and 5 < k_slope < 10

test_peaks()
test_fit()
print("All checks passed")

### Exercise: Timing the recovery

After 30 ms at 0 mV, h has fallen to its steady value there. Compute `h_after`, the value of h after recovery intervals `dts` at -65 mV. Then fit the recovery time constant `tau_rec` by a straight-line fit of ln(h_inf(-65) - h_after) against the interval.

In [ ]:
dts = np.array([1, 2, 4, 6, 8, 12, 16, 24, 32.0])
h_inact = inf_tau(0.0, "h")[0]
hinf_rest, tau_h_rest = inf_tau(-65.0, "h")
h_after = ...
slope = np.polyfit(dts, np.log(hinf_rest - h_after), 1)[0]
tau_rec = ...
print(f"recovery time constant {tau_rec:.2f} ms (tau_h at -65 mV is {tau_h_rest:.2f} ms)")
print("fraction of resting availability:", np.round(h_after / hinf_rest, 2))

In [ ]:
import numpy as np
def test_h():
    "h_after is the value of h after recovering at -65 mV for each interval, starting from h at the end of 30 ms at 0 mV"
    hinf, tau = inf_tau(-65.0, "h"); h0 = inf_tau(0.0, "h")[0]
    assert np.allclose(h_after, hinf - (hinf - h0) * np.exp(-dts / tau), atol=1e-3)

def test_tau():
    "the fitted recovery time constant is within 0.5 ms of tau_h at -65 mV"
    assert abs(tau_rec - inf_tau(-65.0, "h")[1]) < 0.5

test_h()
test_tau()
print("All checks passed")

### Exercise: Measuring the refractory period

`fires(amp, isi)` simulates a spike triggered at 2 ms and a 0.5 ms test pulse of amplitude amp starting isi ms later, and reports whether the test pulse fires a second spike. Complete the bisection that finds the smallest amplitude that fires, for each interval, storing None if even 200 μA/cm² fails.

In [ ]:
import math

def rates_f(V):
    V = V + 1e-7
    return (0.01 * (V + 55) / (1 - math.exp(-(V + 55) / 10)), 0.125 * math.exp(-(V + 65) / 80),
            0.1 * (V + 40) / (1 - math.exp(-(V + 40) / 10)), 4.0 * math.exp(-(V + 65) / 18),
            0.07 * math.exp(-(V + 65) / 20), 1 / (1 + math.exp(-(V + 35) / 10)))

def simulate(I, T, dt=0.02):
    V, n, m, h = -65.0, 0.3177, 0.0529, 0.5961
    out = []
    for k in range(int(T / dt)):
        an, bn, am, bm, ah, bh = rates_f(V)
        dV = I(k * dt) - 120 * m**3 * h * (V - 50) - 36 * n**4 * (V + 77) - 0.3 * (V + 54.387)
        n += dt * (an * (1 - n) - bn * n); m += dt * (am * (1 - m) - bm * m); h += dt * (ah * (1 - h) - bh * h)
        V += dt * dV
        out.append(V)
    return out

def fires(amp, isi):
    V = simulate(lambda t: 20.0 if 2 <= t < 3 else (amp if 2 + isi <= t < 2.5 + isi else 0.0), 2 + isi + 10)
    return max(V[int((2 + isi) / 0.02):]) > 0

def threshold(test):
    lo, hi = 0.0, 200.0
    if not test(hi):
        return None
    for _ in range(10):
        mid = (lo + hi) / 2
        ...
    return hi

rest_threshold = threshold(lambda a: max(simulate(lambda t: a if 2 <= t < 2.5 else 0.0, 12)) > 0)
thresholds = {isi: threshold(lambda a, isi=isi: fires(a, isi)) for isi in (4, 6, 8, 10, 14, 20)}
print(f"threshold at rest: {rest_threshold:.1f} uA/cm^2 for 0.5 ms")
for isi, th in thresholds.items():
    print(f"{isi:2d} ms after the first spike: " + ("no second spike" if th is None else f"{th:6.1f} uA/cm^2 ({th / rest_threshold:.1f} x rest)"))

In [ ]:
import numpy as np
def test_absolute():
    "at 4 ms no pulse up to 200 uA/cm^2 fires a second spike"
    assert thresholds[4] is None

def test_relative():
    "thresholds fall with interval, and at 20 ms are below twice the resting threshold"
    vals = [thresholds[i] for i in (6, 8, 10, 14, 20)]
    assert all(a > b for a, b in zip(vals, vals[1:])) and thresholds[20] < 2 * rest_threshold

test_absolute()
test_relative()
print("All checks passed")

### Exercise: Anode-break excitation

Simulate 50 ms with a hyperpolarising current of amplitude amp from 5 to 25 ms and nothing else, for amp = -2, -5 and -10 μA/cm². Store the peak voltage after the release (from 25 ms on) in `peak[amp]`.

In [ ]:
peak = {}
for amp in (-2, -5, -10):
    V = simulate(lambda t, amp=amp: amp if 5 <= t < 25 else 0.0, 50)
    peak[amp] = ...
    print(f"{amp:3d} uA/cm^2 for 20 ms, then release: peak {peak[amp]:6.1f} mV")

In [ ]:
import numpy as np
def test_rebound():
    "a -5 uA/cm^2 step for 20 ms followed by release produces a spike above 0 mV, while -2 does not"
    assert peak[-5] > 0 and peak[-2] < 0

test_rebound()
print("All checks passed")